# Bai Shopping Brain — Kaggle Auto Train → Eval → Promotion
Включи GPU T4 и нажми Run All. Если reviewed Gold добавлен в Kaggle Inputs, notebook использует его. Если Inputs пусты, создаётся внутренний deterministic bootstrap: 500 train + 60 отдельный holdout. Teacher/model outputs этим путём никогда автоматически не approve-ятся.


In [ ]:
import torch
assert torch.cuda.device_count() >= 1, 'Нужен GPU'
print(torch.cuda.get_device_name(0))


In [ ]:
!pip -q install -U 'transformers>=4.51,<5' 'peft>=0.15,<1' 'datasets>=3,<5' accelerate bitsandbytes sentencepiece
!rm -rf /kaggle/working/tamdeshevle
!git clone --depth 1 --branch main https://github.com/eneonstudio-dev/tamdeshevle.git /kaggle/working/tamdeshevle
%cd /kaggle/working/tamdeshevle


In [ ]:
from pathlib import Path
import subprocess, sys
inputs=Path('/kaggle/input')
evals=list(inputs.rglob('eval-gold.jsonl'))
golds=[p for p in inputs.rglob('gold.jsonl') if p not in evals]
if len(evals)==1 and golds:
    eval_gold=evals[0]; train_gold=sorted(golds); mode='reviewed-input'
else:
    seed=Path('/kaggle/working/bai_seed')
    subprocess.check_call(['node','teacher-lab/training/deterministic-seed.mjs',str(seed)])
    eval_gold=seed/'eval-gold.jsonl'; train_gold=[seed/'gold.jsonl']; mode='deterministic-bootstrap'
cmd=[sys.executable,'teacher-lab/training/kaggle_train_pipeline.py']
for p in train_gold: cmd += ['--gold',str(p)]
cmd += ['--eval-gold',str(eval_gold),'--out','/kaggle/working/bai_auto_train']
print('MODE:',mode); print('TRAIN GOLD:', *train_gold, sep='\n- '); print('EVAL GOLD:',eval_gold)
subprocess.check_call(cmd)


In [ ]:
from pathlib import Path
print(Path('/kaggle/working/bai_auto_train/pipeline-manifest.json').read_text())
print('Artifacts:')
for p in Path('/kaggle/working/bai_auto_train').glob('*.zip'): print('-',p)
